# Chapter 2: Data Loading and Transformation

## Overview

This notebook covers data loading from various sources and performing transformations in Apache Spark.

### Learning Objectives

- Load data from various file formats (CSV, JSON, Parquet)
- Handle schema evolution and schema inference
- Perform data cleaning and validation
- Transform data using built-in functions
- Handle missing values and nulls

## Setup

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, when, coalesce, regexp_replace
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

spark = SparkSession.builder \
    .appName("DataLoadingTransformation") \
    .master("spark://spark-master:7077") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

## Exercise 1: Loading CSV Data

In [ ]:
# Create sample CSV data
csv_data = """id,name,age,salary,department,hire_date\n1,Alice Johnson,28,75000,Engineering,2020-01-15\n2,Bob Smith,35,85000,Marketing,2019-03-22\n3,Charlie Brown,42,95000,Engineering,2018-06-10\n4,Diana Prince,31,72000,Sales,2021-02-28\n5,Eve Davis,29,68000,Marketing,2020-11-05\""\n
# Write to file
with open("/opt/spark/data/employees.csv", "w") as f:\n    f.write(csv_data)

In [ ]:
# Load CSV with schema inference
df_csv = spark.read.csv(\n    "/opt/spark/data/employees.csv",\n    header=True,\n    inferSchema=True\n)\n
df_csv.show()\ndf_csv.printSchema()

## Exercise 2: Data Cleaning - Handling Missing Values

In [ ]:
# Create data with missing values
data_with_nulls = [\n    (1, "Alice", 28, 75000, "Engineering"),\n    (2, "Bob", None, 85000, "Marketing"),\n    (3, None, 42, 95000, "Engineering"),\n    (4, "Diana", 31, None, "Sales"),\n    (5, "Eve", 29, 68000, None)\n]\n
schema = StructType([\n    StructField("id", IntegerType(), True),\n    StructField("name", StringType(), True),\n    StructField("age", IntegerType(), True),\n    StructField("salary", IntegerType(), True),\n    StructField("department", StringType(), True)\n])\n
df_nulls = spark.createDataFrame(data_with_nulls, schema)\nprint("Data with nulls:")\ndf_nulls.show()

In [ ]:
# Drop rows with nulls\ndf_no_nulls = df_nulls.na.drop()\nprint("After dropping nulls:")\ndf_no_nulls.show()\n
# Fill nulls with default values\ndf_filled = df_nulls.na.fill({\n    "name": "Unknown",\n    "age": 0,\n    "salary": 0,\n    "department": "Unassigned"\n})\nprint("After filling nulls:")\ndf_filled.show()

## Exercise 3: Data Transformations

In [ ]:
# String transformations\nfrom pyspark.sql.functions import upper, lower, trim, split\n
df_transformed = df_csv.select(\n    upper(col("name")).alias("name_upper"),\n    lower(col("name")).alias("name_lower"),\n    trim(col("name")).alias("name_trimmed"),\n    split(col("name"), " ")[0].alias("first_name")\n)\n
df_transformed.show()

In [ ]:
# Numeric transformations\nfrom pyspark.sql.functions import round, abs, pow\n
df_numeric = df_csv.select(\n    col("salary"),\n    round(col("salary") / 12, 2).alias("monthly_salary"),\n    (col("salary") * 0.1).alias("bonus"),\n    abs(col("age") - 30).alias("age_diff_from_30")\n)\n
df_numeric.show()

## Cleanup

In [ ]:
spark.stop()\nprint("Spark session stopped.")